# Constructive heuristics (C++)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/constructive-heuristics_cpp.ipynb)

Executable notebook of the chapter “Constructive heuristics” (part “Heuristic algorithms” of the course *Combinatorial Optimization*): it contains the constructive heuristics of the chapter for the knapsack problem, the bin packing problem, $P\,||\,C_{\max}$ and the TSP, written in C++ and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/heuristic-algorithms/constructive-heuristics/). The same notebook is also available in [Python](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/constructive-heuristics_python.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`. Each C++ file is written to the folder `src/` by its `%%writefile` cell; the cell after `Main.cpp` compiles everything with `g++ -std=c++17 -O2` and runs the examples. Nothing to install (Colab already has `g++`).

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## Declarations

The header declares all the functions of the chapter; each function is in its own file, with the same name.

In [1]:
%%writefile src/Constr_heuristics.h
#ifndef CONSTR_HEURISTICS_HEADER
#define CONSTR_HEURISTICS_HEADER

// constructive heuristics for the KP, the BPP, P||Cmax and the TSP
// items, bins, jobs, machines and cities are indexed starting from 0
// ties are always broken in favour of the smallest index

// sorting
void KPOrder(int n, int* p, int* w, int criterion, int* order);   // 0 profit, 1 weight, 2 efficiency
void DecreasingOrder(int n, int* a, int* order);

// KP: returns the profit, x[i] = 1 if item i is selected
int GreedyOrderKP(int n, int c, int* w, int* p, int* order, int* x);

// BPP: returns the number of bins, bin[i] = bin of item i
int FirstFitDecreasing(int n, int c, int* w, int* bin);
int BestFitDecreasing(int n, int c, int* w, int* bin);

// P||Cmax: returns the makespan, mach[j] = machine of job j
int ListScheduling(int n, int m, int* t, int* order, int* mach);
int LPT(int n, int m, int* t, int* mach);

// TSP: d is the n x n distance matrix; every function returns the length of the tour
int TourLength(int n, int** d, int* tour);
int NearestNeighbour(int n, int** d, int s, int* tour);
int MultiStartNN(int n, int** d, int* tour);
int Insertion(int n, int** d, int s, int rule, int* tour);           // 0 NI, 1 FI, 2 CI
int Regret2Insertion(int n, int** d, int s, int* tour);
int PilotNN(int n, int** d, int s, int* tour);

// randomized construction
double ParkMiller(long long* seed);
int RandomizedNN(int n, int** d, int s, int alpha_num, int alpha_den, long long* seed, int* tour);

#endif

Writing src/Constr_heuristics.h


## Sorting

The orders of the KP items and the decreasing order used by FFD, BFD and $LPT$ are computed by insertion sort, which keeps the index order among equal keys. The efficiencies are compared without divisions: $p_i/w_i > p_j/w_j$ if and only if $p_i w_j > p_j w_i$.

In [2]:
%%writefile src/KPOrder.cpp
#include "Constr_heuristics.h"

// order[0..n-1] receives the items sorted by the criterion:
// 0 = non-increasing profit, 1 = non-decreasing weight, 2 = non-increasing efficiency p/w
// insertion sort: items with equal keys keep their index order

static bool Before(int i, int j, int* p, int* w, int criterion)
{
    if (criterion == 0) return p[i] > p[j];
    if (criterion == 1) return w[i] < w[j];
    return p[i] * w[j] > p[j] * w[i];      // p_i/w_i > p_j/w_j without divisions
}

void KPOrder(int n, int* p, int* w, int criterion, int* order)
{
    for (int k = 0; k < n; k++)
    {
        int i = k;                         // item to be inserted in order[0..k-1]
        int pos = k;
        while (pos > 0 && Before(i, order[pos - 1], p, w, criterion))
        {
            order[pos] = order[pos - 1];
            pos--;
        }
        order[pos] = i;
    }
}

Writing src/KPOrder.cpp


In [3]:
%%writefile src/DecreasingOrder.cpp
#include "Constr_heuristics.h"

// order[0..n-1] receives the indices sorted by non-increasing a[i]
// insertion sort: indices with equal values keep their order

void DecreasingOrder(int n, int* a, int* order)
{
    for (int k = 0; k < n; k++)
    {
        int pos = k;
        while (pos > 0 && a[k] > a[order[pos - 1]])
        {
            order[pos] = order[pos - 1];
            pos--;
        }
        order[pos] = k;
    }
}

Writing src/DecreasingOrder.cpp


## The knapsack problem

The greedy algorithm with a given order of the items.

In [4]:
%%writefile src/GreedyOrderKP.cpp
#include "Constr_heuristics.h"

// the items are examined in the order given by order[0..n-1]
// x[i] stores the selection decision for item i
// the function returns the profit of the solution x

int GreedyOrderKP(int n, int c, int* w, int* p, int* order, int* x)
{
    for (int i = 0; i < n; i++)
    {
        x[i] = 0;                          // item i is initially not selected
    }
    int rc = c;                            // residual capacity
    int z = 0;                             // profit of the selected items

    for (int k = 0; k < n; k++)
    {
        int i = order[k];                  // k-th item in the order
        if (w[i] <= rc)                    // feasibility test
        {
            x[i] = 1;                      // select item i
            rc = rc - w[i];                // update residual capacity
            z = z + p[i];
        }
    }
    return z;
}

Writing src/GreedyOrderKP.cpp


## The bin packing problem

First Fit Decreasing and Best Fit Decreasing.

In [5]:
%%writefile src/FirstFitDecreasing.cpp
#include "Constr_heuristics.h"

// the items are sorted by non-increasing weight and packed by First Fit
// the function returns the number of bins used
// bin[i] stores the bin assigned to item i

int FirstFitDecreasing(int n, int c, int* w, int* bin)
{
    int* order = new int[n];
    DecreasingOrder(n, w, order);          // sort: largest items first

    int* rc = new int[n];                  // residual capacities (at most n bins)
    int nb = 0;                            // number of bins opened so far

    for (int k = 0; k < n; k++)
    {
        int i = order[k];                  // k-th largest item
        int s = -1;                        // selected bin (-1 means: not found yet)
        for (int b = 0; b < nb; b++)       // first bin in which item i fits
        {
            if (w[i] <= rc[b])
            {
                s = b;
                break;
            }
        }
        if (s == -1)                       // no bin fits: open a new one
        {
            rc[nb] = c;
            s = nb;
            nb++;
        }
        bin[i] = s;                        // pack item i into bin s
        rc[s] = rc[s] - w[i];
    }

    delete[] rc;
    delete[] order;
    return nb;
}

Writing src/FirstFitDecreasing.cpp


In [6]:
%%writefile src/BestFitDecreasing.cpp
#include "Constr_heuristics.h"

// the items are sorted by non-increasing weight and packed by Best Fit
// the function returns the number of bins used
// bin[i] stores the bin assigned to item i

int BestFitDecreasing(int n, int c, int* w, int* bin)
{
    int* order = new int[n];
    DecreasingOrder(n, w, order);          // sort: largest items first

    int* rc = new int[n];                  // residual capacities (at most n bins)
    int nb = 0;                            // number of bins opened so far

    for (int k = 0; k < n; k++)
    {
        int i = order[k];                  // k-th largest item
        int s = -1;                        // selected bin (-1 means: not found yet)
        for (int b = 0; b < nb; b++)       // fullest bin in which item i fits
        {
            if (w[i] <= rc[b] && (s == -1 || rc[b] < rc[s]))
            {
                s = b;
            }
        }
        if (s == -1)                       // no bin fits: open a new one
        {
            rc[nb] = c;
            s = nb;
            nb++;
        }
        bin[i] = s;                        // pack item i into bin s
        rc[s] = rc[s] - w[i];
    }

    delete[] rc;
    delete[] order;
    return nb;
}

Writing src/BestFitDecreasing.cpp


## Scheduling on parallel machines

List Scheduling and Longest Processing Time first ($LPT$).

In [7]:
%%writefile src/ListScheduling.cpp
#include "Constr_heuristics.h"

// the jobs are examined in the order given by order[0..n-1]
// each job is assigned to the machine with the smallest current load
// the function returns the makespan; mach[j] stores the machine of job j

int ListScheduling(int n, int m, int* t, int* order, int* mach)
{
    int* L = new int[m];                   // loads of the machines
    for (int h = 0; h < m; h++)
    {
        L[h] = 0;
    }

    for (int k = 0; k < n; k++)
    {
        int j = order[k];                  // k-th job of the list
        int s = 0;                         // least loaded machine
        for (int h = 1; h < m; h++)
        {
            if (L[h] < L[s]) s = h;
        }
        mach[j] = s;                       // assign job j to machine s
        L[s] = L[s] + t[j];
    }

    int Cmax = 0;                          // makespan = largest load
    for (int h = 0; h < m; h++)
    {
        if (L[h] > Cmax) Cmax = L[h];
    }
    delete[] L;
    return Cmax;
}

Writing src/ListScheduling.cpp


In [8]:
%%writefile src/LPT.cpp
#include "Constr_heuristics.h"

// Longest Processing Time first: list scheduling with the jobs
// sorted by non-increasing processing time
// the function returns the makespan; mach[j] stores the machine of job j

int LPT(int n, int m, int* t, int* mach)
{
    int* order = new int[n];
    DecreasingOrder(n, t, order);          // longest jobs first
    int Cmax = ListScheduling(n, m, t, order, mach);
    delete[] order;
    return Cmax;
}

Writing src/LPT.cpp


## The travelling salesman problem

The distances are stored in an $n\times n$ matrix `d`, and a tour in an array of $n$ cities. In `Insertion` the three selection rules share the same loop: the value of a candidate is its distance from the subtour ($NI$), the same distance with the sign changed, so that the minimum corresponds to the farthest city ($FI$), or its minimum insertion cost ($CI$).

In [9]:
%%writefile src/TourLength.cpp
#include "Constr_heuristics.h"

// length of the tour tour[0], tour[1], ..., tour[n-1], tour[0]

int TourLength(int n, int** d, int* tour)
{
    int L = 0;
    for (int k = 0; k < n - 1; k++)
    {
        L = L + d[tour[k]][tour[k + 1]];
    }
    return L + d[tour[n - 1]][tour[0]];    // closing edge
}

Writing src/TourLength.cpp


In [10]:
%%writefile src/NearestNeighbour.cpp
#include "Constr_heuristics.h"

// starting from city s, the tour is extended at its end with the
// nearest city not yet visited
// the function returns the length of the tour stored in tour[0..n-1]

int NearestNeighbour(int n, int** d, int s, int* tour)
{
    bool* visited = new bool[n];
    for (int i = 0; i < n; i++)
    {
        visited[i] = false;
    }
    tour[0] = s;                           // the tour starts at city s
    visited[s] = true;

    for (int len = 1; len < n; len++)
    {
        int last = tour[len - 1];          // last city of the partial tour
        int k = -1;                        // nearest unvisited city
        for (int j = 0; j < n; j++)
        {
            if (!visited[j] && (k == -1 || d[last][j] < d[last][k])) k = j;
        }
        tour[len] = k;                     // append k at the end
        visited[k] = true;
    }

    delete[] visited;
    return TourLength(n, d, tour);
}

Writing src/NearestNeighbour.cpp


In [11]:
%%writefile src/MultiStartNN.cpp
#include "Constr_heuristics.h"

// runs the nearest neighbour heuristic from every city
// and keeps the shortest tour (the first one found, in case of ties)
// the function returns its length; the tour is stored in tour[0..n-1]

int MultiStartNN(int n, int** d, int* tour)
{
    int* cur = new int[n];
    int best = -1;                         // length of the best tour found
    for (int s = 0; s < n; s++)
    {
        int L = NearestNeighbour(n, d, s, cur);
        if (best == -1 || L < best)
        {
            best = L;
            for (int k = 0; k < n; k++) tour[k] = cur[k];
        }
    }
    delete[] cur;
    return best;
}

Writing src/MultiStartNN.cpp


In [12]:
%%writefile src/Insertion.cpp
#include "Constr_heuristics.h"

// insertion heuristics: starting from the subtour (s), the city selected by
// the rule is inserted into the subtour where it increases the length the least
// rule: 0 = nearest insertion (NI), 1 = farthest insertion (FI), 2 = cheapest insertion (CI)
// the function returns the length of the tour stored in tour[0..n-1]

// cheapest position for city k in the subtour tour[0..len-1]:
// returns the increase of length, and in *pos the edge (tour[a], tour[a+1]) to break
static int BestPosition(int len, int** d, int* tour, int k, int* pos)
{
    int best = -1;
    for (int a = 0; a < len; a++)
    {
        int i = tour[a], j = tour[(a + 1) % len];
        int cost = d[i][k] + d[k][j] - d[i][j];
        if (best == -1 || cost < best)
        {
            best = cost;
            *pos = a;
        }
    }
    return best;
}

int Insertion(int n, int** d, int s, int rule, int* tour)
{
    bool* in = new bool[n];                // in[i] = true if city i is in the subtour
    for (int i = 0; i < n; i++)
    {
        in[i] = false;
    }
    tour[0] = s;                           // the subtour (s)
    in[s] = true;

    for (int len = 1; len < n; len++)
    {
        // selection step: city k to be inserted
        int k = -1, keyk = 0, pos = 0;
        for (int q = 0; q < n; q++)
        {
            if (in[q]) continue;
            int key;                       // value of city q for the rule
            if (rule == 2)
            {
                int a;
                key = BestPosition(len, d, tour, q, &a);   // insertion cost
            }
            else
            {
                key = d[q][tour[0]];       // distance from the subtour
                for (int a = 1; a < len; a++)
                {
                    if (d[q][tour[a]] < key) key = d[q][tour[a]];
                }
                if (rule == 1) key = -key; // farthest: maximize the distance
            }
            if (k == -1 || key < keyk)
            {
                k = q;
                keyk = key;
            }
        }
        // insertion step: k goes where it increases the length the least
        BestPosition(len, d, tour, k, &pos);
        for (int a = len; a > pos + 1; a--)
        {
            tour[a] = tour[a - 1];
        }
        tour[pos + 1] = k;
        in[k] = true;
    }

    delete[] in;
    return TourLength(n, d, tour);
}

Writing src/Insertion.cpp


## Randomized construction

The Park–Miller generator and the randomized nearest neighbour. The threshold of the RCL is tested in integer arithmetic, with $\alpha$ given as a fraction $\alpha_{num}/\alpha_{den}$, so that no rounding error can change the list.

In [13]:
%%writefile src/ParkMiller.cpp
#include "Constr_heuristics.h"

// Park-Miller minimal standard generator:
// seed <- 16807 * seed mod (2^31 - 1), and u = seed / (2^31 - 1) in (0,1)

double ParkMiller(long long* seed)
{
    *seed = (16807LL * (*seed)) % 2147483647LL;
    return (double)(*seed) / 2147483647.0;
}

Writing src/ParkMiller.cpp


In [14]:
%%writefile src/RandomizedNN.cpp
#include "Constr_heuristics.h"

// randomized nearest neighbour with a value-based restricted candidate list:
// RCL = { j unvisited : d[last][j] <= dmin + alpha (dmax - dmin) },
// with alpha = alpha_num / alpha_den; the next city is drawn uniformly from the RCL
// (a random number is used only when the RCL has at least two cities)
// the function returns the length of the tour stored in tour[0..n-1]

int RandomizedNN(int n, int** d, int s, int alpha_num, int alpha_den,
                 long long* seed, int* tour)
{
    bool* visited = new bool[n];
    int* RCL = new int[n];
    for (int i = 0; i < n; i++)
    {
        visited[i] = false;
    }
    tour[0] = s;
    visited[s] = true;

    for (int len = 1; len < n; len++)
    {
        int last = tour[len - 1];
        int dmin = -1, dmax = -1;          // smallest and largest distance from last
        for (int j = 0; j < n; j++)
        {
            if (visited[j]) continue;
            if (dmin == -1 || d[last][j] < dmin) dmin = d[last][j];
            if (dmax == -1 || d[last][j] > dmax) dmax = d[last][j];
        }
        int size = 0;                      // restricted candidate list
        for (int j = 0; j < n; j++)
        {
            // d <= dmin + alpha (dmax - dmin), in integer arithmetic
            if (!visited[j] && alpha_den * d[last][j] <= alpha_den * dmin + alpha_num * (dmax - dmin))
            {
                RCL[size] = j;
                size++;
            }
        }
        int k = RCL[0];
        if (size > 1)
        {
            double u = ParkMiller(seed);   // random number in (0,1)
            k = RCL[(int)(u * size)];      // uniform choice in the RCL
        }
        tour[len] = k;
        visited[k] = true;
    }

    delete[] RCL;
    delete[] visited;
    return TourLength(n, d, tour);
}

Writing src/RandomizedNN.cpp


## Regret and look-ahead

Regret-2 insertion and the pilot method with the nearest neighbour.

In [15]:
%%writefile src/Regret2Insertion.cpp
#include "Constr_heuristics.h"

// regret-2 insertion: starting from the subtour (s), insert the city with the largest
// regret = (second cheapest insertion cost) - (cheapest insertion cost),
// ties broken by the smaller cheapest cost, then by the smaller index,
// at its cheapest position
// the function returns the length of the tour stored in tour[0..n-1]

int Regret2Insertion(int n, int** d, int s, int* tour)
{
    bool* in = new bool[n];
    for (int i = 0; i < n; i++)
    {
        in[i] = false;
    }
    tour[0] = s;
    in[s] = true;

    for (int len = 1; len < n; len++)
    {
        int k = -1, regk = 0, c1k = 0, posk = 0;
        for (int q = 0; q < n; q++)
        {
            if (in[q]) continue;
            int c1 = -1, c2 = -1, pos = 0; // cheapest and second cheapest cost
            for (int a = 0; a < len; a++)
            {
                int i = tour[a], j = tour[(a + 1) % len];
                int cost = d[i][q] + d[q][j] - d[i][j];
                if (c1 == -1 || cost < c1)
                {
                    c2 = c1;
                    c1 = cost;
                    pos = a;
                }
                else if (c2 == -1 || cost < c2)
                {
                    c2 = cost;
                }
            }
            if (c2 == -1) c2 = c1;         // a single position: no regret
            int reg = c2 - c1;
            if (k == -1 || reg > regk || (reg == regk && c1 < c1k))
            {
                k = q;
                regk = reg;
                c1k = c1;
                posk = pos;
            }
        }
        for (int a = len; a > posk + 1; a--)   // insert k after position posk
        {
            tour[a] = tour[a - 1];
        }
        tour[posk + 1] = k;
        in[k] = true;
    }

    delete[] in;
    return TourLength(n, d, tour);
}

Writing src/Regret2Insertion.cpp


In [16]:
%%writefile src/PilotNN.cpp
#include "Constr_heuristics.h"

// pilot method with the nearest neighbour heuristic as pilot:
// every candidate next city is evaluated by completing the partial tour
// with nearest neighbour; the candidate with the shortest completed tour is appended
// the function returns the length of the tour stored in tour[0..n-1]

int PilotNN(int n, int** d, int s, int* tour)
{
    bool* visited = new bool[n];
    int* trial = new int[n];               // partial tour completed by the pilot
    bool* used = new bool[n];
    for (int i = 0; i < n; i++)
    {
        visited[i] = false;
    }
    tour[0] = s;
    visited[s] = true;

    for (int len = 1; len < n; len++)
    {
        int k = -1, bestL = -1;
        for (int q = 0; q < n; q++)        // candidate next city q
        {
            if (visited[q]) continue;
            for (int i = 0; i < n; i++) used[i] = visited[i];
            for (int a = 0; a < len; a++) trial[a] = tour[a];
            trial[len] = q;
            used[q] = true;
            for (int t = len + 1; t < n; t++)  // pilot: nearest neighbour completion
            {
                int last = trial[t - 1], next = -1;
                for (int j = 0; j < n; j++)
                {
                    if (!used[j] && (next == -1 || d[last][j] < d[last][next])) next = j;
                }
                trial[t] = next;
                used[next] = true;
            }
            int L = TourLength(n, d, trial);
            if (k == -1 || L < bestL)
            {
                k = q;
                bestL = L;
            }
        }
        tour[len] = k;                     // append the best candidate
        visited[k] = true;
    }

    delete[] used;
    delete[] trial;
    delete[] visited;
    return TourLength(n, d, tour);
}

Writing src/PilotNN.cpp


## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [17]:
%%writefile src/Main.cpp
#include <cstdio>
#include "Constr_heuristics.h"

// runs every example of the lecture notes and every instance of the exercises,
// and compares the solutions found with those of an independent Python implementation
// (this file is generated by a script)

static const int e1[] = {1,1,0,0};
static const int e2[] = {0,0,1,1};
static const int e3[] = {1,0,1,0};
static const int e4[] = {0,1,0,0,0};
static const int e5[] = {1,0,0,1,0};
static const int e6[] = {0,0,1,1,0};
static const int e7[] = {1,0,0,0,1};
static const int e8[] = {0,1,0,1,1};
static const int e9[] = {1,1,0,0,0};
static const int e10[] = {0,1,0,0,0};
static const int e11[] = {0,0,1,0,1};
static const int e12[] = {1,0,0,0,1};
static const int e13[] = {1,2,0,1,0,2};
static const int e14[] = {1,2,0,1,0,2};
static const int e15[] = {1,1,0,1,2,0};
static const int e16[] = {1,1,0,1,2,0};
static const int e17[] = {0,1,2,2,0,2,1};
static const int e18[] = {0,1,2,2,0,2,1};
static const int e19[] = {2,0,1,2,0,2,1,3};
static const int e20[] = {2,0,1,2,0,2,1,3};
static const int e21[] = {1,2,1,0,0,2,2};
static const int e22[] = {1,2,1,0,2,0,2};
static const int e23[] = {0,1,2,2,0,1,0};
static const int e24[] = {0,2,1,0,2,1,0};
static const int e25[] = {0,1,2,2,0,1,0};
static const int e26[] = {0,1,2,0,2,0,1,0};
static const int e27[] = {2,1,1,2,2,0,0,0};
static const int e28[] = {0,1,2,0,2,1,0};
static const int e29[] = {1,2,2,1,0,0,1};
static const int e30[] = {0,1,2,0,1,2,0};
static const int e31[] = {2,0,1,1,0,2,1};
static const int e32[] = {0,2,1,3,4};
static const int e33[] = {1,3,0,2,4};
static const int e34[] = {2,0,3,1,4};
static const int e35[] = {3,0,2,1,4};
static const int e36[] = {4,1,3,0,2};
static const int e37[] = {1,3,0,2,4};
static const int e38[] = {0,3,1,4,2};
static const int e39[] = {0,3,1,4,2};
static const int e40[] = {0,3,1,4,2};
static const int e41[] = {1,4,2,0,3};
static const int e42[] = {1,4,2,0,3};
static const int e43[] = {1,4,2,0,3};
static const int e44[] = {2,4,1,3,0};
static const int e45[] = {2,0,3,1,4};
static const int e46[] = {2,4,1,3,0};
static const int e47[] = {3,1,4,2,0};
static const int e48[] = {3,0,2,4,1};
static const int e49[] = {3,1,4,2,0};
static const int e50[] = {4,2,0,3,1};
static const int e51[] = {4,1,3,0,2};
static const int e52[] = {4,2,0,3,1};
static const int e53[] = {0,3,1,4,2};
static const int e54[] = {0,4,1,3,2};
static const int e55[] = {0,2,1,3,4};
static const int e56[] = {0,3,1,4,2};
static const int e57[] = {0,2,1,3,4};
static const int e58[] = {0,3,2,1,4};
static const int e59[] = {0,5,2,1,3,4};
static const int e60[] = {1,2,5,0,3,4};
static const int e61[] = {2,1,5,0,3,4};
static const int e62[] = {3,0,5,2,1,4};
static const int e63[] = {4,5,0,2,1,3};
static const int e64[] = {5,0,2,1,3,4};
static const int e65[] = {1,2,5,0,3,4};
static const int e66[] = {0,4,1,2,3,5};
static const int e67[] = {0,5,4,1,2,3};
static const int e68[] = {0,3,1,2,4,5};
static const int e69[] = {0,3,2,1,4,5};
static const int e70[] = {0,3,1,2,4,5};
static const int e71[] = {0,4,2,1,5,3};
static const int e72[] = {1,2,5,3,4,0};
static const int e73[] = {2,1,5,3,4,0};
static const int e74[] = {3,5,2,1,4,0};
static const int e75[] = {4,0,1,2,5,3};
static const int e76[] = {5,3,2,1,4,0};
static const int e77[] = {1,2,5,3,4,0};
static const int e78[] = {0,1,2,5,3,4};
static const int e79[] = {0,1,2,5,3,4};
static const int e80[] = {0,1,2,5,3,4};
static const int e81[] = {0,4,1,5,3,2};
static const int e82[] = {1,5,0,4,2,3};
static const int e83[] = {2,0,4,1,5,3};
static const int e84[] = {3,0,4,1,5,2};
static const int e85[] = {4,0,1,5,3,2};
static const int e86[] = {5,1,0,4,2,3};
static const int e87[] = {4,0,1,5,3,2};
static const int e88[] = {0,5,1,4,2,3};
static const int e89[] = {0,1,5,3,2,4};
static const int e90[] = {0,2,3,5,1,4};
static const int e91[] = {0,1,3,2,4};
static const int e92[] = {1,0,3,2,4};
static const int e93[] = {2,3,0,1,4};
static const int e94[] = {3,2,4,1,0};
static const int e95[] = {4,2,3,0,1};
static const int e96[] = {1,0,3,2,4};
static const int e97[] = {0,4,2,3,1};
static const int e98[] = {0,1,4,2,3};
static const int e99[] = {0,4,2,3,1};
static const int e100[] = {0,1,2,3,4};
static const int e101[] = {0,3,2,4,1};

static int errors = 0, checks = 0;

static void check(const char* instance, const char* algorithm, int found, int expected)
{
    checks++;
    if (found != expected) errors++;
    printf("%-18s %-18s %4d %s\n", instance, algorithm, found,
           found == expected ? "OK" : "<-- EXPECTED A DIFFERENT VALUE");
}

static void checkVec(const char* instance, const char* what, int* found, const int* expected, int n)
{
    checks++;
    bool ok = true;
    for (int k = 0; k < n; k++) if (found[k] != expected[k]) ok = false;
    if (!ok) errors++;
    printf("%-18s %-18s ", instance, what);
    for (int k = 0; k < n; k++) printf("%d ", found[k] + 1);
    printf("%s\n", ok ? "OK" : "<-- EXPECTED A DIFFERENT SOLUTION");
}

int main()
{
    int x[16], order[16], bin[16], mach[16], tour[16];
    (void)order;
    { int p[] = {10,7,6,4}; int w[] = {5,4,3,3};
      KPOrder(4, p, w, 0, order);
      check("KP reference", "greedy profit", GreedyOrderKP(4, 9, w, p, order, x), 17);
      checkVec("KP reference", "  x", x, e1, 4);
      KPOrder(4, p, w, 1, order);
      check("KP reference", "greedy weight", GreedyOrderKP(4, 9, w, p, order, x), 10);
      checkVec("KP reference", "  x", x, e2, 4);
      KPOrder(4, p, w, 2, order);
      check("KP reference", "greedy efficiency", GreedyOrderKP(4, 9, w, p, order, x), 16);
      checkVec("KP reference", "  x", x, e3, 4);
    }
    { int p[] = {4,15,11,8,2}; int w[] = {3,8,7,2,5};
      KPOrder(5, p, w, 0, order);
      check("EX_KO_1", "greedy profit", GreedyOrderKP(5, 9, w, p, order, x), 15);
      checkVec("EX_KO_1", "  x", x, e4, 5);
      KPOrder(5, p, w, 1, order);
      check("EX_KO_1", "greedy weight", GreedyOrderKP(5, 9, w, p, order, x), 12);
      checkVec("EX_KO_1", "  x", x, e5, 5);
      KPOrder(5, p, w, 2, order);
      check("EX_KO_1", "greedy efficiency", GreedyOrderKP(5, 9, w, p, order, x), 19);
      checkVec("EX_KO_1", "  x", x, e6, 5);
    }
    { int p[] = {9,4,8,6,7}; int w[] = {7,2,9,5,6};
      KPOrder(5, p, w, 0, order);
      check("EX_KO_2", "greedy profit", GreedyOrderKP(5, 13, w, p, order, x), 16);
      checkVec("EX_KO_2", "  x", x, e7, 5);
      KPOrder(5, p, w, 1, order);
      check("EX_KO_2", "greedy weight", GreedyOrderKP(5, 13, w, p, order, x), 17);
      checkVec("EX_KO_2", "  x", x, e8, 5);
      KPOrder(5, p, w, 2, order);
      check("EX_KO_2", "greedy efficiency", GreedyOrderKP(5, 13, w, p, order, x), 13);
      checkVec("EX_KO_2", "  x", x, e9, 5);
    }
    { int p[] = {8,13,5,12,7}; int w[] = {5,9,4,8,2};
      KPOrder(5, p, w, 0, order);
      check("EX_KO_3", "greedy profit", GreedyOrderKP(5, 10, w, p, order, x), 13);
      checkVec("EX_KO_3", "  x", x, e10, 5);
      KPOrder(5, p, w, 1, order);
      check("EX_KO_3", "greedy weight", GreedyOrderKP(5, 10, w, p, order, x), 12);
      checkVec("EX_KO_3", "  x", x, e11, 5);
      KPOrder(5, p, w, 2, order);
      check("EX_KO_3", "greedy efficiency", GreedyOrderKP(5, 10, w, p, order, x), 15);
      checkVec("EX_KO_3", "  x", x, e12, 5);
    }
    { int w[] = {4,4,5,3,2,3};
      check("BPP reference", "FFD", FirstFitDecreasing(6, 7, w, bin), 3);
      checkVec("BPP reference", "  bin", bin, e13, 6);
      check("BPP reference", "BFD", BestFitDecreasing(6, 7, w, bin), 3);
      checkVec("BPP reference", "  bin", bin, e14, 6);
    }
    { int w[] = {2,2,3,2,2,3};
      check("BPP FF<FFD", "FFD", FirstFitDecreasing(6, 7, w, bin), 3);
      checkVec("BPP FF<FFD", "  bin", bin, e15, 6);
      check("BPP FF<FFD", "BFD", BestFitDecreasing(6, 7, w, bin), 3);
      checkVec("BPP FF<FFD", "  bin", bin, e16, 6);
    }
    { int w[] = {3,3,5,2,6,2,6};
      check("EX_BD_1", "FFD", FirstFitDecreasing(7, 10, w, bin), 3);
      checkVec("EX_BD_1", "  bin", bin, e17, 7);
      check("EX_BD_1", "BFD", BestFitDecreasing(7, 10, w, bin), 3);
      checkVec("EX_BD_1", "  bin", bin, e18, 7);
    }
    { int w[] = {2,6,3,5,4,2,6,2};
      check("EX_BD_2", "FFD", FirstFitDecreasing(8, 10, w, bin), 4);
      checkVec("EX_BD_2", "  bin", bin, e19, 8);
      check("EX_BD_2", "BFD", BestFitDecreasing(8, 10, w, bin), 4);
      checkVec("EX_BD_2", "  bin", bin, e20, 8);
    }
    { int w[] = {4,4,5,7,2,2,4};
      check("EX_BD_3", "FFD", FirstFitDecreasing(7, 10, w, bin), 3);
      checkVec("EX_BD_3", "  bin", bin, e21, 7);
      check("EX_BD_3", "BFD", BestFitDecreasing(7, 10, w, bin), 3);
      checkVec("EX_BD_3", "  bin", bin, e22, 7);
    }
    { int t[] = {5,5,4,4,3,3,3};
      { int o[] = {0,1,2,3,4,5,6}; check("P||Cmax reference", "LIST input", ListScheduling(7, 3, t, o, mach), 11);
        checkVec("P||Cmax reference", "  mach", mach, e23, 7); }
      { int o[] = {6,5,4,3,2,1,0}; check("P||Cmax reference", "LIST reverse", ListScheduling(7, 3, t, o, mach), 12);
        checkVec("P||Cmax reference", "  mach", mach, e24, 7); }
      check("P||Cmax reference", "LPT", LPT(7, 3, t, mach), 11);
      checkVec("P||Cmax reference", "  mach", mach, e25, 7);
    }
    { int t[] = {2,8,7,6,8,3,4,9};
      { int o[] = {0,1,2,3,4,5,6,7}; check("EX_LPT_1", "LIST input", ListScheduling(8, 3, t, o, mach), 20);
        checkVec("EX_LPT_1", "  mach", mach, e26, 8); }
      check("EX_LPT_1", "LPT", LPT(8, 3, t, mach), 16);
      checkVec("EX_LPT_1", "  mach", mach, e27, 8);
    }
    { int t[] = {3,6,5,4,4,9,7};
      { int o[] = {0,1,2,3,4,5,6}; check("EX_LPT_2", "LIST input", ListScheduling(7, 3, t, o, mach), 15);
        checkVec("EX_LPT_2", "  mach", mach, e28, 7); }
      check("EX_LPT_2", "LPT", LPT(7, 3, t, mach), 14);
      checkVec("EX_LPT_2", "  mach", mach, e29, 7);
    }
    { int t[] = {5,8,8,4,5,7,4};
      { int o[] = {0,1,2,3,4,5,6}; check("EX_LPT_3", "LIST input", ListScheduling(7, 3, t, o, mach), 15);
        checkVec("EX_LPT_3", "  mach", mach, e30, 7); }
      check("EX_LPT_3", "LPT", LPT(7, 3, t, mach), 16);
      checkVec("EX_LPT_3", "  mach", mach, e31, 7);
    }
    { int dd[][5] = {{0,5,2,2,9},{5,0,4,3,4},{2,4,0,4,7},{2,3,4,0,7},{9,4,7,7,0}};
      int* d[5]; for (int i = 0; i < 5; i++) d[i] = dd[i];
      check("TSP reference", "NN from 1", NearestNeighbour(5, d, 0, tour), 25);
      checkVec("TSP reference", "  tour", tour, e32, 5);
      check("TSP reference", "NN from 2", NearestNeighbour(5, d, 1, tour), 18);
      checkVec("TSP reference", "  tour", tour, e33, 5);
      check("TSP reference", "NN from 3", NearestNeighbour(5, d, 2, tour), 18);
      checkVec("TSP reference", "  tour", tour, e34, 5);
      check("TSP reference", "NN from 4", NearestNeighbour(5, d, 3, tour), 19);
      checkVec("TSP reference", "  tour", tour, e35, 5);
      check("TSP reference", "NN from 5", NearestNeighbour(5, d, 4, tour), 18);
      checkVec("TSP reference", "  tour", tour, e36, 5);
      check("TSP reference", "multi-start NN", MultiStartNN(5, d, tour), 18);
      checkVec("TSP reference", "  tour", tour, e37, 5);
      check("TSP reference", "NI from 1", Insertion(5, d, 0, 0, tour), 18);
      checkVec("TSP reference", "  tour", tour, e38, 5);
      check("TSP reference", "FI from 1", Insertion(5, d, 0, 1, tour), 18);
      checkVec("TSP reference", "  tour", tour, e39, 5);
      check("TSP reference", "CI from 1", Insertion(5, d, 0, 2, tour), 18);
      checkVec("TSP reference", "  tour", tour, e40, 5);
      check("TSP reference", "NI from 2", Insertion(5, d, 1, 0, tour), 18);
      checkVec("TSP reference", "  tour", tour, e41, 5);
      check("TSP reference", "FI from 2", Insertion(5, d, 1, 1, tour), 18);
      checkVec("TSP reference", "  tour", tour, e42, 5);
      check("TSP reference", "CI from 2", Insertion(5, d, 1, 2, tour), 18);
      checkVec("TSP reference", "  tour", tour, e43, 5);
      check("TSP reference", "NI from 3", Insertion(5, d, 2, 0, tour), 18);
      checkVec("TSP reference", "  tour", tour, e44, 5);
      check("TSP reference", "FI from 3", Insertion(5, d, 2, 1, tour), 18);
      checkVec("TSP reference", "  tour", tour, e45, 5);
      check("TSP reference", "CI from 3", Insertion(5, d, 2, 2, tour), 18);
      checkVec("TSP reference", "  tour", tour, e46, 5);
      check("TSP reference", "NI from 4", Insertion(5, d, 3, 0, tour), 18);
      checkVec("TSP reference", "  tour", tour, e47, 5);
      check("TSP reference", "FI from 4", Insertion(5, d, 3, 1, tour), 18);
      checkVec("TSP reference", "  tour", tour, e48, 5);
      check("TSP reference", "CI from 4", Insertion(5, d, 3, 2, tour), 18);
      checkVec("TSP reference", "  tour", tour, e49, 5);
      check("TSP reference", "NI from 5", Insertion(5, d, 4, 0, tour), 18);
      checkVec("TSP reference", "  tour", tour, e50, 5);
      check("TSP reference", "FI from 5", Insertion(5, d, 4, 1, tour), 18);
      checkVec("TSP reference", "  tour", tour, e51, 5);
      check("TSP reference", "CI from 5", Insertion(5, d, 4, 2, tour), 18);
      checkVec("TSP reference", "  tour", tour, e52, 5);
      check("TSP reference", "regret-2 from 1", Regret2Insertion(5, d, 0, tour), 18);
      checkVec("TSP reference", "  tour", tour, e53, 5);
      check("TSP reference", "pilot NN from 1", PilotNN(5, d, 0, tour), 22);
      checkVec("TSP reference", "  tour", tour, e54, 5);
      long long seed = 1;
      check("TSP reference", "RCL run 1", RandomizedNN(5, d, 0, 3, 10, &seed, tour), 25);
      checkVec("TSP reference", "  tour", tour, e55, 5);
      check("TSP reference", "RCL run 2", RandomizedNN(5, d, 0, 3, 10, &seed, tour), 18);
      checkVec("TSP reference", "  tour", tour, e56, 5);
      check("TSP reference", "RCL run 3", RandomizedNN(5, d, 0, 3, 10, &seed, tour), 25);
      checkVec("TSP reference", "  tour", tour, e57, 5);
      check("TSP reference", "RCL run 4", RandomizedNN(5, d, 0, 3, 10, &seed, tour), 23);
      checkVec("TSP reference", "  tour", tour, e58, 5);
    }
    { int dd[][6] = {{0,5,4,4,5,2},{5,0,1,5,5,4},{4,1,0,4,5,3},{4,5,4,0,9,6},{5,5,5,9,0,3},{2,4,3,6,3,0}};
      int* d[6]; for (int i = 0; i < 6; i++) d[i] = dd[i];
      check("TSP 6 cities", "NN from 1", NearestNeighbour(6, d, 0, tour), 25);
      checkVec("TSP 6 cities", "  tour", tour, e59, 6);
      check("TSP 6 cities", "NN from 2", NearestNeighbour(6, d, 1, tour), 24);
      checkVec("TSP 6 cities", "  tour", tour, e60, 6);
      check("TSP 6 cities", "NN from 3", NearestNeighbour(6, d, 2, tour), 25);
      checkVec("TSP 6 cities", "  tour", tour, e61, 6);
      check("TSP 6 cities", "NN from 4", NearestNeighbour(6, d, 3, tour), 24);
      checkVec("TSP 6 cities", "  tour", tour, e62, 6);
      check("TSP 6 cities", "NN from 5", NearestNeighbour(6, d, 4, tour), 24);
      checkVec("TSP 6 cities", "  tour", tour, e63, 6);
      check("TSP 6 cities", "NN from 6", NearestNeighbour(6, d, 5, tour), 24);
      checkVec("TSP 6 cities", "  tour", tour, e64, 6);
      check("TSP 6 cities", "multi-start NN", MultiStartNN(6, d, tour), 24);
      checkVec("TSP 6 cities", "  tour", tour, e65, 6);
      check("TSP 6 cities", "NI from 1", Insertion(6, d, 0, 0, tour), 23);
      checkVec("TSP 6 cities", "  tour", tour, e66, 6);
      check("TSP 6 cities", "FI from 1", Insertion(6, d, 0, 1, tour), 19);
      checkVec("TSP 6 cities", "  tour", tour, e67, 6);
      check("TSP 6 cities", "CI from 1", Insertion(6, d, 0, 2, tour), 20);
      checkVec("TSP 6 cities", "  tour", tour, e68, 6);
      check("TSP 6 cities", "regret-2 from 1", Regret2Insertion(6, d, 0, tour), 19);
      checkVec("TSP 6 cities", "  tour", tour, e69, 6);
      check("TSP 6 cities", "pilot NN from 1", PilotNN(6, d, 0, tour), 20);
      checkVec("TSP 6 cities", "  tour", tour, e70, 6);
    }
    { int dd[][6] = {{0,7,7,9,4,9},{7,0,2,5,6,4},{7,2,0,3,5,2},{9,5,3,0,5,1},{4,6,5,5,0,6},{9,4,2,1,6,0}};
      int* d[6]; for (int i = 0; i < 6; i++) d[i] = dd[i];
      check("EX_TC_1", "NN from 1", NearestNeighbour(6, d, 0, tour), 25);
      checkVec("EX_TC_1", "  tour", tour, e71, 6);
      check("EX_TC_1", "NN from 2", NearestNeighbour(6, d, 1, tour), 21);
      checkVec("EX_TC_1", "  tour", tour, e72, 6);
      check("EX_TC_1", "NN from 3", NearestNeighbour(6, d, 2, tour), 23);
      checkVec("EX_TC_1", "  tour", tour, e73, 6);
      check("EX_TC_1", "NN from 4", NearestNeighbour(6, d, 3, tour), 24);
      checkVec("EX_TC_1", "  tour", tour, e74, 6);
      check("EX_TC_1", "NN from 5", NearestNeighbour(6, d, 4, tour), 21);
      checkVec("EX_TC_1", "  tour", tour, e75, 6);
      check("EX_TC_1", "NN from 6", NearestNeighbour(6, d, 5, tour), 25);
      checkVec("EX_TC_1", "  tour", tour, e76, 6);
      check("EX_TC_1", "multi-start NN", MultiStartNN(6, d, tour), 21);
      checkVec("EX_TC_1", "  tour", tour, e77, 6);
      check("EX_TC_1", "NI from 1", Insertion(6, d, 0, 0, tour), 21);
      checkVec("EX_TC_1", "  tour", tour, e78, 6);
      check("EX_TC_1", "FI from 1", Insertion(6, d, 0, 1, tour), 21);
      checkVec("EX_TC_1", "  tour", tour, e79, 6);
      check("EX_TC_1", "CI from 1", Insertion(6, d, 0, 2, tour), 21);
      checkVec("EX_TC_1", "  tour", tour, e80, 6);
    }
    { int dd[][6] = {{0,3,4,4,2,3},{3,0,7,6,4,2},{4,7,0,4,4,6},{4,6,4,0,6,4},{2,4,4,6,0,5},{3,2,6,4,5,0}};
      int* d[6]; for (int i = 0; i < 6; i++) d[i] = dd[i];
      check("EX_TC_2", "NN from 1", NearestNeighbour(6, d, 0, tour), 20);
      checkVec("EX_TC_2", "  tour", tour, e81, 6);
      check("EX_TC_2", "NN from 2", NearestNeighbour(6, d, 1, tour), 21);
      checkVec("EX_TC_2", "  tour", tour, e82, 6);
      check("EX_TC_2", "NN from 3", NearestNeighbour(6, d, 2, tour), 20);
      checkVec("EX_TC_2", "  tour", tour, e83, 6);
      check("EX_TC_2", "NN from 4", NearestNeighbour(6, d, 3, tour), 22);
      checkVec("EX_TC_2", "  tour", tour, e84, 6);
      check("EX_TC_2", "NN from 5", NearestNeighbour(6, d, 4, tour), 19);
      checkVec("EX_TC_2", "  tour", tour, e85, 6);
      check("EX_TC_2", "NN from 6", NearestNeighbour(6, d, 5, tour), 19);
      checkVec("EX_TC_2", "  tour", tour, e86, 6);
      check("EX_TC_2", "multi-start NN", MultiStartNN(6, d, tour), 19);
      checkVec("EX_TC_2", "  tour", tour, e87, 6);
      check("EX_TC_2", "NI from 1", Insertion(6, d, 0, 0, tour), 21);
      checkVec("EX_TC_2", "  tour", tour, e88, 6);
      check("EX_TC_2", "FI from 1", Insertion(6, d, 0, 1, tour), 19);
      checkVec("EX_TC_2", "  tour", tour, e89, 6);
      check("EX_TC_2", "CI from 1", Insertion(6, d, 0, 2, tour), 20);
      checkVec("EX_TC_2", "  tour", tour, e90, 6);
    }
    { int dd[][5] = {{0,3,4,3,6},{3,0,4,3,5},{4,4,0,1,3},{3,3,1,0,4},{6,5,3,4,0}};
      int* d[5]; for (int i = 0; i < 5; i++) d[i] = dd[i];
      check("EX_TC_3", "NN from 1", NearestNeighbour(5, d, 0, tour), 16);
      checkVec("EX_TC_3", "  tour", tour, e91, 5);
      check("EX_TC_3", "NN from 2", NearestNeighbour(5, d, 1, tour), 15);
      checkVec("EX_TC_3", "  tour", tour, e92, 5);
      check("EX_TC_3", "NN from 3", NearestNeighbour(5, d, 2, tour), 15);
      checkVec("EX_TC_3", "  tour", tour, e93, 5);
      check("EX_TC_3", "NN from 4", NearestNeighbour(5, d, 3, tour), 15);
      checkVec("EX_TC_3", "  tour", tour, e94, 5);
      check("EX_TC_3", "NN from 5", NearestNeighbour(5, d, 4, tour), 15);
      checkVec("EX_TC_3", "  tour", tour, e95, 5);
      check("EX_TC_3", "multi-start NN", MultiStartNN(5, d, tour), 15);
      checkVec("EX_TC_3", "  tour", tour, e96, 5);
      check("EX_TC_3", "NI from 1", Insertion(5, d, 0, 0, tour), 16);
      checkVec("EX_TC_3", "  tour", tour, e97, 5);
      check("EX_TC_3", "FI from 1", Insertion(5, d, 0, 1, tour), 15);
      checkVec("EX_TC_3", "  tour", tour, e98, 5);
      check("EX_TC_3", "CI from 1", Insertion(5, d, 0, 2, tour), 16);
      checkVec("EX_TC_3", "  tour", tour, e99, 5);
      long long seed = 2026;
      check("EX_TC_3", "RCL run 1", RandomizedNN(5, d, 0, 1, 2, &seed, tour), 18);
      checkVec("EX_TC_3", "  tour", tour, e100, 5);
      check("EX_TC_3", "RCL run 2", RandomizedNN(5, d, 0, 1, 2, &seed, tour), 15);
      checkVec("EX_TC_3", "  tour", tour, e101, 5);
    }
    printf("\n%d checks, %d errors\n", checks, errors);
    return errors == 0 ? 0 : 1;
}

Writing src/Main.cpp


In [18]:
!g++ -std=c++17 -O2 -o CONSTR_HEUR src/*.cpp && ./CONSTR_HEUR

KP reference       greedy profit        17 OK
KP reference         x                2 2 1 1 OK
KP reference       greedy weight        10 OK
KP reference         x                1 1 2 2 OK
KP reference       greedy efficiency    16 OK
KP reference         x                2 1 2 1 OK
EX_KO_1            greedy profit        15 OK
EX_KO_1              x                1 2 1 1 1 OK
EX_KO_1            greedy weight        12 OK
EX_KO_1              x                2 1 1 2 1 OK
EX_KO_1            greedy efficiency    19 OK
EX_KO_1              x                1 1 2 2 1 OK
EX_KO_2            greedy profit        16 OK
EX_KO_2              x                2 1 1 1 2 OK
EX_KO_2            greedy weight        17 OK
EX_KO_2              x                1 2 1 2 2 OK
EX_KO_2            greedy efficiency    13 OK
EX_KO_2              x                2 2 1 1 1 OK
EX_KO_3            greedy profit        13 OK
EX_KO_3              x                1 2 1 1 1 OK
EX_KO_3            greedy weight    

## Try it yourself

Change the data in the cell below and run it again.

In [19]:
%%writefile try.cpp
#include <cstdio>
#include "src/Constr_heuristics.h"

// Try it yourself: change the data below, compile and run again
// g++ -std=c++17 -O2 -o try try.cpp $(ls src/*.cpp | grep -v Main.cpp)

static void show(int n, int* a)          // items, bins, machines, cities numbered from 1
{
    for (int k = 0; k < n; k++) printf(k == 0 ? "%d" : " %d", a[k] + 1);
    printf("\n");
}

static void showTour(const char* name, int L, int n, int* tour)
{
    printf("TSP %-18s length %3d, tour ", name, L);
    show(n, tour);
}

int main()
{
    // change the data below and run again
    const int n = 5;                     // number of items
    int w[n] = {5, 4, 3, 3, 2};          // weights of the items (KP and BPP)
    int p[n] = {10, 7, 6, 4, 3};         // profits of the items (KP)
    int c = 9;                           // capacity of the knapsack and of the bins
    const int nj = 7;                    // number of jobs
    int t[nj] = {5, 5, 4, 4, 3, 3, 3};   // processing times of the jobs
    int m = 3;                           // number of machines
    const int nc = 6;                    // number of cities
    int dd[nc][nc] = {{0, 5, 2, 2, 9, 6}, {5, 0, 4, 3, 4, 3}, {2, 4, 0, 4, 7, 5},
                      {2, 3, 4, 0, 7, 4}, {9, 4, 7, 7, 0, 2}, {6, 3, 5, 4, 2, 0}};   // distances
    int* d[nc];
    for (int i = 0; i < nc; i++) d[i] = dd[i];

    int order[n], x[n], bin[n], mach[nj], tour[nc], sel[n];
    KPOrder(n, p, w, 2, order);
    int z = GreedyOrderKP(n, c, w, p, order, x);
    int ns = 0;                          // selected items
    for (int i = 0; i < n; i++) if (x[i] == 1) sel[ns++] = i;
    printf("KP greedy by efficiency: profit %d, selected items ", z);
    show(ns, sel);
    int nb = FirstFitDecreasing(n, c, w, bin);
    printf("BPP first fit decreasing: %d bins, bin of each item ", nb);
    show(n, bin);
    nb = BestFitDecreasing(n, c, w, bin);
    printf("BPP best fit decreasing:  %d bins, bin of each item ", nb);
    show(n, bin);
    int cmax = LPT(nj, m, t, mach);
    printf("P||Cmax LPT: makespan %d, machine of each job ", cmax);
    show(nj, mach);

    int L = NearestNeighbour(nc, d, 0, tour);
    showTour("nearest neighbour", L, nc, tour);
    L = MultiStartNN(nc, d, tour);
    showTour("multi-start NN", L, nc, tour);
    L = Insertion(nc, d, 0, 1, tour);
    showTour("farthest insertion", L, nc, tour);
    L = Insertion(nc, d, 0, 2, tour);
    showTour("cheapest insertion", L, nc, tour);
    L = Regret2Insertion(nc, d, 0, tour);
    showTour("regret-2 insertion", L, nc, tour);
    L = PilotNN(nc, d, 0, tour);
    showTour("pilot NN", L, nc, tour);
    long long seed = 1;
    L = RandomizedNN(nc, d, 0, 3, 10, &seed, tour);
    showTour("randomized NN", L, nc, tour);
    return 0;
}

Writing try.cpp


In [20]:
!g++ -std=c++17 -O2 -o try try.cpp $(ls src/*.cpp | grep -v Main.cpp) && ./try

KP greedy by efficiency: profit 16, selected items 1 3
BPP first fit decreasing: 2 bins, bin of each item 1 1 2 2 2
BPP best fit decreasing:  2 bins, bin of each item 1 1 2 2 2
P||Cmax LPT: makespan 11, machine of each job 1 2 3 3 1 2 1
TSP nearest neighbour  length  24, tour 1 3 2 4 6 5
TSP multi-start NN     length  18, tour 2 4 1 3 6 5
TSP farthest insertion length  18, tour 1 4 2 5 6 3
TSP cheapest insertion length  18, tour 1 4 6 5 2 3
TSP regret-2 insertion length  18, tour 1 4 2 5 6 3
TSP pilot NN           length  18, tour 1 4 6 5 2 3
TSP randomized NN      length  20, tour 1 3 2 6 5 4


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).